## 0. Install Dependencies
Install required packages: DuckDB, PyArrow, pandas, and tqdm.

In [1]:
#123

%pip install -q duckdb pyarrow pandas tqdm

Note: you may need to restart the kernel to use updated packages.


## 1. Imports
Import standard library modules and third-party packages.

In [2]:
from __future__ import annotations

import io
import os
import re
import sys
import json
import math
import shutil
import zipfile
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import duckdb
import pandas as pd
from tqdm.auto import tqdm

## 2. Configuration
Set all path constants, DuckDB settings, and ingestion parameters in one place. Adjust `DATA_ROOT` and `DUCKDB_MEMORY_LIMIT` to match your machine.

In [3]:
# Root folder that contains the Freddie Mac yearly ZIPs
DATA_ROOT = Path(r"C:\Users\websi\OneDrive - UT Cloud\Semester\4. SS2026\MA5_10 Masterarbeit (30 ECTS)\data\freddie_mac_sflld_data")

# Output folders
PARQUET_ROOT = DATA_ROOT / "parquet"
ORIGINATION_OUT = PARQUET_ROOT / "origination"
PERFORMANCE_OUT = PARQUET_ROOT / "performance"

# Temporary extraction folder
TEMP_ROOT = DATA_ROOT / "_tmp_ingest"
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

# DuckDB database file for the ingestion session
DUCKDB_PATH = DATA_ROOT / "freddie_mac_ingest.duckdb"

# Behavior
OVERWRITE_EXISTING = False
MAX_THREADS = max(1, (os.cpu_count() or 4) - 1)
DUCKDB_MEMORY_LIMIT = "24GB"   # adjust if needed
PARQUET_COMPRESSION = "ZSTD"
PARQUET_ROW_GROUP_SIZE = 250_000

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"DUCKDB_PATH: {DUCKDB_PATH}")
print(f"MAX_THREADS: {MAX_THREADS}")

DATA_ROOT: C:\Users\websi\OneDrive - UT Cloud\Semester\4. SS2026\MA5_10 Masterarbeit (30 ECTS)\data\freddie_mac_sflld_data
DUCKDB_PATH: C:\Users\websi\OneDrive - UT Cloud\Semester\4. SS2026\MA5_10 Masterarbeit (30 ECTS)\data\freddie_mac_sflld_data\freddie_mac_ingest.duckdb
MAX_THREADS: 7


## 3. Column Schemas
Define the expected column names for origination (32 cols) and performance (32 cols) files as specified by the Freddie Mac SFLLD data dictionary. Assertions guard against schema drift.

In [ ]:
ORIGINATION_COLS = [
    "credit_score",
    "first_payment_date",
    "first_time_homebuyer_flag",
    "maturity_date",
    "msa_or_metropolitan_division",
    "mortgage_insurance_percentage",
    "number_of_units",
    "occupancy_status",
    "original_combined_loan_to_value",
    "original_debt_to_income_ratio",
    "original_upb",
    "original_loan_to_value",
    "original_interest_rate",
    "channel",
    "prepayment_penalty_mortgage_flag",
    "amortization_type",
    "property_state",
    "property_type",
    "postal_code",
    "loan_sequence_number",
    "loan_purpose",
    "original_loan_term",
    "number_of_borrowers",
    "seller_name",
    "servicer_name",
    "super_conforming_flag",
    "pre_relief_refinance_loan_sequence_number",
    "special_eligibility_program",
    "relief_refinance_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "mi_cancellation_indicator",
]

PERFORMANCE_COLS = [
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_non_interest_bearing_upb",
    "ddlpi",
    "mi_recoveries",
    "net_sale_proceeds",
    "non_mi_recoveries",
    "expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "miscellaneous_expenses",
    "actual_loss_calculation",
    "cumulative_modification_cost",
    "interest_rate_step_indicator",
    "payment_deferral_flag",
    "estimated_loan_to_value",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_status_code",
    "current_month_modification_cost",
    "interest_bearing_upb",
]

assert len(ORIGINATION_COLS) == 32
assert len(PERFORMANCE_COLS) == 32

## 4. File Patterns & SQL Helper Functions
Regex patterns to locate yearly ZIPs and quarterly sub-ZIPs. Helper functions for SQL expression building (`TRY_CAST`, `TRY_STRPTIME`, `NULLIF/TRIM`) and output path construction partitioned by `vintage_year / vintage_quarter`.

In [ ]:
YEAR_ZIP_PATTERN = re.compile(r"historical_data_(\d{4})\.zip$", re.IGNORECASE)
QUARTER_ZIP_PATTERN = re.compile(r"historical_data_(\d{4})Q([1-4])\.zip$", re.IGNORECASE)
ORIG_TXT_PATTERN = re.compile(r"historical_data_(\d{4})Q([1-4])\.txt$", re.IGNORECASE)
PERF_TXT_PATTERN = re.compile(r"historical_data_time_(\d{4})Q([1-4])\.txt$", re.IGNORECASE)


def find_year_zip_files(root: Path) -> list[Path]:
    """
    Find yearly ZIP archives like historical_data_1999.zip.
    """
    files = sorted([p for p in root.iterdir() if p.is_file() and YEAR_ZIP_PATTERN.search(p.name)])
    if not files:
        raise FileNotFoundError(
            f"No yearly ZIP files matching historical_data_YYYY.zip found in {root}"
        )
    return files


def ensure_dirs():
    ORIGINATION_OUT.mkdir(parents=True, exist_ok=True)
    PERFORMANCE_OUT.mkdir(parents=True, exist_ok=True)
    TEMP_ROOT.mkdir(parents=True, exist_ok=True)


def output_path(dataset: str, year: int, quarter: str) -> Path:
    base = ORIGINATION_OUT if dataset == "origination" else PERFORMANCE_OUT
    out_dir = base / f"vintage_year={year}" / f"vintage_quarter={quarter}"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f"part_{year}{quarter}.parquet"


def trim_sql(expr: str) -> str:
    return f"NULLIF(TRIM({expr}), '')"


def yyyymm_to_date_sql(expr: str) -> str:
    """
    Convert YYYYMM strings to DATE (first day of month), safely.
    """
    cleaned = trim_sql(expr)
    return f"TRY_STRPTIME({cleaned} || '01', '%Y%m%d')"


def int_sql(expr: str) -> str:
    return f"TRY_CAST({trim_sql(expr)} AS INTEGER)"


def dbl_sql(expr: str) -> str:
    return f"TRY_CAST({trim_sql(expr)} AS DOUBLE)"


def str_sql(expr: str) -> str:
    return trim_sql(expr)


def build_raw_column_map(n_cols: int) -> dict[str, str]:
    return {f"c{i+1}": "VARCHAR" for i in range(n_cols)}

## 5. Origination SELECT Builder
Generates the DuckDB `SELECT` clause that casts all 32 raw `VARCHAR` columns to their typed counterparts (INTEGER, DOUBLE, DATE) for the origination file. Provenance columns (`vintage_year`, `vintage_quarter`, `source_*`, `ingested_at_utc`) are appended.

In [ ]:
def build_origination_select(year: int, quarter: str, source_year_zip: str, source_quarter_zip: str, source_txt_name: str) -> str:
    c = lambda i: f"c{i}"
    return f"""
    SELECT
        {int_sql(c(1))}  AS credit_score,
        {yyyymm_to_date_sql(c(2))} AS first_payment_date,
        {str_sql(c(3))}  AS first_time_homebuyer_flag,
        {yyyymm_to_date_sql(c(4))} AS maturity_date,
        {int_sql(c(5))}  AS msa_or_metropolitan_division,
        {int_sql(c(6))}  AS mortgage_insurance_percentage,
        {int_sql(c(7))}  AS number_of_units,
        {str_sql(c(8))}  AS occupancy_status,
        {int_sql(c(9))}  AS original_combined_loan_to_value,
        {int_sql(c(10))} AS original_debt_to_income_ratio,
        {dbl_sql(c(11))} AS original_upb,
        {int_sql(c(12))} AS original_loan_to_value,
        {dbl_sql(c(13))} AS original_interest_rate,
        {str_sql(c(14))} AS channel,
        {str_sql(c(15))} AS prepayment_penalty_mortgage_flag,
        {str_sql(c(16))} AS amortization_type,
        {str_sql(c(17))} AS property_state,
        {str_sql(c(18))} AS property_type,
        {int_sql(c(19))} AS postal_code,
        {str_sql(c(20))} AS loan_sequence_number,
        {str_sql(c(21))} AS loan_purpose,
        {int_sql(c(22))} AS original_loan_term,
        {int_sql(c(23))} AS number_of_borrowers,
        {str_sql(c(24))} AS seller_name,
        {str_sql(c(25))} AS servicer_name,
        {str_sql(c(26))} AS super_conforming_flag,
        {str_sql(c(27))} AS pre_relief_refinance_loan_sequence_number,
        {str_sql(c(28))} AS special_eligibility_program,
        {str_sql(c(29))} AS relief_refinance_indicator,
        {int_sql(c(30))} AS property_valuation_method,
        {str_sql(c(31))} AS interest_only_indicator,
        {str_sql(c(32))} AS mi_cancellation_indicator,

        {year} AS vintage_year,
        '{quarter}' AS vintage_quarter,
        '{year}{quarter}' AS vintage,
        '{source_year_zip}' AS source_year_zip,
        '{source_quarter_zip}' AS source_quarter_zip,
        '{source_txt_name}' AS source_txt_name,
        CURRENT_TIMESTAMP AS ingested_at_utc
    """

## 6. Performance SELECT Builder
Same pattern as the origination builder but for the 32 performance columns. Note: `current_loan_delinquency_status` is intentionally kept as VARCHAR because Freddie Mac uses non-numeric codes (`XX`, `RA`, `RM`) alongside integer values.

In [ ]:
def build_performance_select(year: int, quarter: str, source_year_zip: str, source_quarter_zip: str, source_txt_name: str) -> str:
    c = lambda i: f"c{i}"
    return f"""
    SELECT
        {str_sql(c(1))}  AS loan_sequence_number,
        {yyyymm_to_date_sql(c(2))} AS monthly_reporting_period,
        {dbl_sql(c(3))}  AS current_actual_upb,
        {str_sql(c(4))}  AS current_loan_delinquency_status,
        {int_sql(c(5))}  AS loan_age,
        {int_sql(c(6))}  AS remaining_months_to_legal_maturity,
        {yyyymm_to_date_sql(c(7))} AS defect_settlement_date,
        {str_sql(c(8))}  AS modification_flag,
        {str_sql(c(9))}  AS zero_balance_code,
        {yyyymm_to_date_sql(c(10))} AS zero_balance_effective_date,
        {dbl_sql(c(11))} AS current_interest_rate,
        {dbl_sql(c(12))} AS current_non_interest_bearing_upb,
        {yyyymm_to_date_sql(c(13))} AS ddlpi,
        {dbl_sql(c(14))} AS mi_recoveries,
        {str_sql(c(15))} AS net_sale_proceeds,
        {dbl_sql(c(16))} AS non_mi_recoveries,
        {dbl_sql(c(17))} AS expenses,
        {dbl_sql(c(18))} AS legal_costs,
        {dbl_sql(c(19))} AS maintenance_and_preservation_costs,
        {dbl_sql(c(20))} AS taxes_and_insurance,
        {dbl_sql(c(21))} AS miscellaneous_expenses,
        {dbl_sql(c(22))} AS actual_loss_calculation,
        {dbl_sql(c(23))} AS cumulative_modification_cost,
        {str_sql(c(24))} AS interest_rate_step_indicator,
        {str_sql(c(25))} AS payment_deferral_flag,
        {int_sql(c(26))} AS estimated_loan_to_value,
        {dbl_sql(c(27))} AS zero_balance_removal_upb,
        {dbl_sql(c(28))} AS delinquent_accrued_interest,
        {str_sql(c(29))} AS delinquency_due_to_disaster,
        {str_sql(c(30))} AS borrower_assistance_status_code,
        {dbl_sql(c(31))} AS current_month_modification_cost,
        {dbl_sql(c(32))} AS interest_bearing_upb,

        {year} AS vintage_year,
        '{quarter}' AS vintage_quarter,
        '{year}{quarter}' AS vintage,
        '{source_year_zip}' AS source_year_zip,
        '{source_quarter_zip}' AS source_quarter_zip,
        '{source_txt_name}' AS source_txt_name,
        CURRENT_TIMESTAMP AS ingested_at_utc
    """

## 7. DuckDB Connection
Open a persistent DuckDB connection with thread count and memory limit configured.

In [5]:
def connect_duckdb(db_path: Path) -> duckdb.DuckDBPyConnection:
    con = duckdb.connect(str(db_path))
    con.execute(f"PRAGMA threads={MAX_THREADS}")
    con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    con.execute("PRAGMA enable_progress_bar")
    return con

## 8. ZIP Extraction Utility
Streams a single member out of an open `ZipFile` to a temporary directory. Uses an 8 MB copy buffer to avoid loading large TXT files entirely into memory.

In [ ]:
def extract_member_to_temp(zf: zipfile.ZipFile, member_name: str, temp_dir: Path) -> Path:
    """
    Extract one ZIP member to a temporary file and return its path.
    """
    target = temp_dir / Path(member_name).name
    with zf.open(member_name) as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
    return target

## 9. Core TXT to Parquet Ingestion Function
Reads one pipe-delimited TXT file via DuckDB `read_csv` (all columns as raw VARCHAR, `sample_size=-1` for full-file type inference, `ignore_errors=False` so bad rows surface immediately) and writes a ZSTD-compressed Parquet file with 250k-row groups. Skips already-existing output files when `OVERWRITE_EXISTING = False`.

In [ ]:
def ingest_txt_with_duckdb(
    con: duckdb.DuckDBPyConnection,
    txt_path: Path,
    out_path: Path,
    dataset: str,
    year: int,
    quarter: str,
    source_year_zip: str,
    source_quarter_zip: str,
    source_txt_name: str,
) -> None:
    """
    Read a Freddie TXT with DuckDB and write a single parquet file.
    """
    if out_path.exists() and not OVERWRITE_EXISTING:
        return

    n_cols = 32
    raw_columns = build_raw_column_map(n_cols)

    select_sql = (
        build_origination_select(year, quarter, source_year_zip, source_quarter_zip, source_txt_name)
        if dataset == "origination"
        else build_performance_select(year, quarter, source_year_zip, source_quarter_zip, source_txt_name)
    )

    raw_csv = f"""
        read_csv(
            '{txt_path.as_posix()}',
            delim='|',
            header=False,
            columns={json.dumps(raw_columns)},
            quote='',
            escape='',
            null_padding=True,
            sample_size=-1,
            ignore_errors=False
        )
    """

    copy_sql = f"""
        COPY (
            {select_sql}
            FROM {raw_csv}
        )
        TO '{out_path.as_posix()}'
        (
            FORMAT PARQUET,
            COMPRESSION {PARQUET_COMPRESSION},
            ROW_GROUP_SIZE {PARQUET_ROW_GROUP_SIZE}
        )
    """

    con.execute(copy_sql)

## 10. Quarter ZIP Inspection
Parse a quarter ZIP file to locate the origination and performance TXT files and extract year/quarter metadata.

In [ ]:
def inspect_inner_quarter_zip(inner_zip_bytes: bytes, inner_zip_name: str) -> dict:
    """
    Inspect one quarter ZIP and identify origination/performance TXT files.
    """
    result = {
        "inner_zip_name": inner_zip_name,
        "year": None,
        "quarter": None,
        "orig_txt": None,
        "perf_txt": None,
    }

    m = QUARTER_ZIP_PATTERN.search(Path(inner_zip_name).name)
    if not m:
        raise ValueError(f"Unexpected quarter ZIP name: {inner_zip_name}")

    result["year"] = int(m.group(1))
    result["quarter"] = f"Q{m.group(2)}"

    with zipfile.ZipFile(io.BytesIO(inner_zip_bytes), "r") as qzip:
        names = qzip.namelist()
        for n in names:
            base = Path(n).name
            if ORIG_TXT_PATTERN.search(base):
                result["orig_txt"] = n
            elif PERF_TXT_PATTERN.search(base):
                result["perf_txt"] = n

    if result["orig_txt"] is None or result["perf_txt"] is None:
        raise ValueError(
            f"Could not identify both TXT files inside {inner_zip_name}. "
            f"Found orig={result['orig_txt']}, perf={result['perf_txt']}"
        )

    return result

## 11. Build Ingestion Manifest
Iterates over all yearly ZIPs in `DATA_ROOT`, opens each one, inspects every quarterly sub-ZIP, and records source paths plus intended output paths into a manifest DataFrame. No data is read or written here — this is a pure discovery scan.

In [ ]:
ensure_dirs()
year_zips = find_year_zip_files(DATA_ROOT)

manifest_rows = []

for year_zip_path in tqdm(year_zips, desc="Scanning yearly ZIPs", unit="year_zip"):
    with zipfile.ZipFile(year_zip_path, "r") as yz:
        inner_names = sorted(
            [
                n for n in yz.namelist()
                if QUARTER_ZIP_PATTERN.search(Path(n).name)
            ]
        )
        for inner_name in inner_names:
            with yz.open(inner_name) as f:
                inner_bytes = f.read()
            meta = inspect_inner_quarter_zip(inner_bytes, inner_name)

            manifest_rows.append({
                "source_year_zip": year_zip_path.name,
                "source_year_zip_path": str(year_zip_path),
                "source_quarter_zip": Path(inner_name).name,
                "year": meta["year"],
                "quarter": meta["quarter"],
                "orig_txt": Path(meta["orig_txt"]).name,
                "perf_txt": Path(meta["perf_txt"]).name,
                "orig_out": str(output_path("origination", meta["year"], meta["quarter"])),
                "perf_out": str(output_path("performance", meta["year"], meta["quarter"])),
            })

manifest = pd.DataFrame(manifest_rows).sort_values(["year", "quarter"]).reset_index(drop=True)
manifest

## 12. Inspect Manifest
Preview the manifest and print summary statistics: total quarter count and year range.

In [ ]:
display(manifest.head(12))
print(f"Quarter ZIPs found: {len(manifest)}")
print(f"Years covered: {manifest['year'].min()} - {manifest['year'].max()}")
print(manifest.groupby("year").size().tail(10))

## 13. Ingestion Pipeline Function
Iterate over the manifest year-by-year, extract TXT files from nested ZIPs, and write them to partitioned Parquet.

In [ ]:
def run_ingestion(manifest: pd.DataFrame) -> None:
    con = connect_duckdb(DUCKDB_PATH)

    try:
        year_groups = list(manifest.groupby("year", sort=True))

        for year, year_df in tqdm(year_groups, desc="Ingesting years", unit="year"):
            year_zip_path = Path(year_df["source_year_zip_path"].iloc[0])

            with zipfile.ZipFile(year_zip_path, "r") as year_zip:
                quarter_records = year_df.to_dict(orient="records")

                for rec in tqdm(
                    quarter_records,
                    desc=f"Year {year}",
                    unit="quarter",
                    leave=False
                ):
                    qzip_name = rec["source_quarter_zip"]
                    vintage_year = int(rec["year"])
                    vintage_quarter = rec["quarter"]

                    orig_out = Path(rec["orig_out"])
                    perf_out = Path(rec["perf_out"])

                    if (
                        orig_out.exists()
                        and perf_out.exists()
                        and not OVERWRITE_EXISTING
                    ):
                        continue

                    with year_zip.open(qzip_name) as inner_f:
                        inner_bytes = inner_f.read()

                    with zipfile.ZipFile(io.BytesIO(inner_bytes), "r") as quarter_zip:
                        temp_q_dir = TEMP_ROOT / f"{vintage_year}{vintage_quarter}"
                        temp_q_dir.mkdir(parents=True, exist_ok=True)

                        try:
                            orig_txt_tmp = extract_member_to_temp(quarter_zip, rec["orig_txt"], temp_q_dir)
                            perf_txt_tmp = extract_member_to_temp(quarter_zip, rec["perf_txt"], temp_q_dir)

                            file_bar = tqdm(
                                total=2,
                                desc=f"{vintage_year}{vintage_quarter}",
                                unit="file",
                                leave=False
                            )

                            if not (orig_out.exists() and not OVERWRITE_EXISTING):
                                ingest_txt_with_duckdb(
                                    con=con,
                                    txt_path=orig_txt_tmp,
                                    out_path=orig_out,
                                    dataset="origination",
                                    year=vintage_year,
                                    quarter=vintage_quarter,
                                    source_year_zip=rec["source_year_zip"],
                                    source_quarter_zip=rec["source_quarter_zip"],
                                    source_txt_name=rec["orig_txt"],
                                )
                            file_bar.update(1)

                            if not (perf_out.exists() and not OVERWRITE_EXISTING):
                                ingest_txt_with_duckdb(
                                    con=con,
                                    txt_path=perf_txt_tmp,
                                    out_path=perf_out,
                                    dataset="performance",
                                    year=vintage_year,
                                    quarter=vintage_quarter,
                                    source_year_zip=rec["source_year_zip"],
                                    source_quarter_zip=rec["source_quarter_zip"],
                                    source_txt_name=rec["perf_txt"],
                                )
                            file_bar.update(1)
                            file_bar.close()

                        finally:
                            shutil.rmtree(temp_q_dir, ignore_errors=True)

    finally:
        con.close()

## 14. Run Ingestion
Execute the full ingestion pipeline over all quarters in the manifest.

In [ ]:
run_ingestion(manifest)

## 15. Verify Output Files
List and count the generated Parquet files for origination and performance datasets.

In [ ]:
orig_files = sorted(ORIGINATION_OUT.rglob("*.parquet"))
perf_files = sorted(PERFORMANCE_OUT.rglob("*.parquet"))

print(f"Origination parquet files: {len(orig_files)}")
print(f"Performance parquet files: {len(perf_files)}")

if orig_files:
    print("First origination file:", orig_files[0])
if perf_files:
    print("First performance file:", perf_files[0])

## 16. Row Count Validation
Query all Parquet files with DuckDB to confirm the total number of ingested origination and performance rows.

In [6]:
con = connect_duckdb(DUCKDB_PATH)

orig_glob = (ORIGINATION_OUT / "**" / "*.parquet").as_posix()
perf_glob = (PERFORMANCE_OUT / "**" / "*.parquet").as_posix()

orig_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{orig_glob}', union_by_name=True)").fetchone()[0]
perf_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{perf_glob}', union_by_name=True)").fetchone()[0]

print(f"Origination rows: {orig_count:,}")
print(f"Performance rows: {perf_count:,}")

Origination rows: 48,597,637
Performance rows: 2,800,558,227


## 17. Repartition Performance Data by Reporting Period
**Why this step is necessary:** The ingestion pipeline (cells 9–14) partitions performance files by *origination vintage* (`vintage_year / vintage_quarter`). This is correct for the ingestion source structure, but all downstream thesis queries filter performance data by `monthly_reporting_period` (e.g. `WHERE monthly_reporting_period BETWEEN '2007-01-01' AND '2009-12-31'` for the subprime crisis regime). Under the vintage partition, DuckDB must scan every single vintage-quarter file to answer such a query, because a loan originated in 1999 has performance observations in 2007. With ~100 vintage-quarter partitions this is very slow.

This cell performs a single full-scan repartition into a new folder `parquet/performance_by_period/reporting_year=YYYY/` using DuckDB's built-in `PARTITION_BY`. After this step, regime-period queries touch only the 2–3 relevant year-partitions. The original vintage-partitioned files are left untouched.

> **Note:** This is a one-time operation. On reruns `OVERWRITE_OR_IGNORE TRUE` skips already-written partition files.

In [7]:
# Output folder — sits alongside the existing vintage-partitioned performance folder
PERFORMANCE_BY_PERIOD_OUT = PARQUET_ROOT / "performance_by_period"
PERFORMANCE_BY_PERIOD_OUT.mkdir(parents=True, exist_ok=True)

perf_by_period_glob = (PERFORMANCE_BY_PERIOD_OUT / "**" / "*.parquet").as_posix()

con = connect_duckdb(DUCKDB_PATH)

try:
    print("Starting repartition — reading all vintage-partitioned performance files...")
    print("This is a full table scan (~1B+ rows). Expect 10–30 minutes depending on hardware.")

    con.execute(f"""
        COPY (
            SELECT
                *,
                YEAR(monthly_reporting_period) AS reporting_year
            FROM read_parquet('{perf_glob}', union_by_name=True)
        )
        TO '{PERFORMANCE_BY_PERIOD_OUT.as_posix()}'
        (
            FORMAT          PARQUET,
            PARTITION_BY    (reporting_year),
            COMPRESSION     {PARQUET_COMPRESSION},
            ROW_GROUP_SIZE  {PARQUET_ROW_GROUP_SIZE},
            OVERWRITE_OR_IGNORE TRUE
        )
    """)
    print("Repartition complete.")
finally:
    con.close()

# --- Verification ---
period_partitions = sorted({p.parent.name for p in PERFORMANCE_BY_PERIOD_OUT.rglob('*.parquet')})
period_files      = sorted(PERFORMANCE_BY_PERIOD_OUT.rglob('*.parquet'))

print(f"Reporting-year partitions written : {len(period_partitions)}")
print(f"Total parquet files               : {len(period_files)}")
print(f"Partitions                        : {period_partitions}")


Starting repartition — reading all vintage-partitioned performance files...
This is a full table scan (~1B+ rows). Expect 10–30 minutes depending on hardware.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Repartition complete.
Reporting-year partitions written : 27
Total parquet files               : 27
Partitions                        : ['reporting_year=1999', 'reporting_year=2000', 'reporting_year=2001', 'reporting_year=2002', 'reporting_year=2003', 'reporting_year=2004', 'reporting_year=2005', 'reporting_year=2006', 'reporting_year=2007', 'reporting_year=2008', 'reporting_year=2009', 'reporting_year=2010', 'reporting_year=2011', 'reporting_year=2012', 'reporting_year=2013', 'reporting_year=2014', 'reporting_year=2015', 'reporting_year=2016', 'reporting_year=2017', 'reporting_year=2018', 'reporting_year=2019', 'reporting_year=2020', 'reporting_year=2021', 'reporting_year=2022', 'reporting_year=2023', 'reporting_year=2024', 'reporting_year=2025']
